# 강의 02 · 실습 2 — 구조화 출력 · (4) 고난도 I

## 1. 문제상황

- 회사의 주간 회의는 개발팀과 운영팀이 같은 시간에 같은 방에서 진행하고, 회의록 담당자 한 사람이 두 팀의 메모를 한 문서에 이어서 적습니다.
- 메모에는 [개발팀] [운영팀]처럼 팀 이름 줄이 있고, 그 아래에 그 팀의 결정사항·할 일·미결이 섞여 적혀 있습니다.
- 문서 전체를 한 덩어리로 요약하는 요약기는 어느 할 일이 어느 팀의 것인지 잃어버립니다.
- 팀장에게 팀별로 나눈 보고서를 보내야 하는데, 어느 할 일과 미결이 어느 팀의 것인지 요약만으로는 알 수 없어 담당자가 원문을 다시 읽고 손으로 갈라 적습니다.
- 팀이 늘어나면 담당자가 문서를 팀별로 잘라 요약기를 여러 번 돌려야 하고, 잘못 자르면 팀 하나가 통째로 빠집니다.

## 2. 문제와 목표

- **문제**: 한 문서에 여러 팀의 메모가 섞여 있는데, 요약 스키마가 팀 단위를 표현하지 못합니다. 팀 정보가 사라지고, 팀이 빠져도 아무도 모릅니다.
- **목표**: 팀 하나의 요약을 나타내는 클래스 안에 회의 요약 클래스 `MeetingSummary`(결정사항·할 일·미결)를 넣고, 그 팀 요약의 리스트를 가지는 주간 보고서 클래스를 선언합니다. 모델을 한 번 호출해 문서 전체를 팀별로 나눈 보고서 객체로 받고, 팀이 비어 있거나 팀 이름이 겹치면 검증에서 걸리게 합니다. 받은 객체에서 팀별 미결 항목을 한 목록으로 모읍니다.
    - 팀 요약: 팀 이름과 그 팀의 회의 요약(결정사항·할 일·미결).
    - 주간 보고서: 주차와 팀 요약 리스트.
    - 검증에서 걸리는 경우: 팀 리스트가 비어 있거나 팀 이름이 겹칠 때.
- **목표 달성 여부의 판정 기준**: 두 팀의 메모를 넣었을 때 팀별로 나뉜 주간 보고서 객체가 돌아오고(팀 수 2), 각 팀의 할 일이 그 팀 이름 아래에 묶여 있으며, 팀 이름이 겹치는 입력과 팀이 비어 있는 입력이 검증 오류를 내는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec02_ex02_s4_diagram.svg)

## 4. 단계별 요구사항

1. **중첩 스키마를 선언합니다.**
    - 할 일 한 건 `ActionItem`(`task`·`owner`·`due`)과 회의 요약 `MeetingSummary`(`decisions`·`action_items`·`open_questions`)를 선언하고, 팀 이름(`team`)과 그 팀의 요약(`summary: MeetingSummary`)을 가지는 `TeamSummary`, 주차(`week`)와 팀 요약 리스트(`teams: list[TeamSummary]`)를 가지는 `WeeklyReport`를 선언합니다.
2. **인스턴스 전체를 보는 검증기를 답니다.**
    - `WeeklyReport`에 `@model_validator(mode="after")`를 붙여, `teams`가 비어 있거나 팀 이름이 겹치면 `ValueError`를 냅니다.
3. **스키마를 지정해 모델을 호출합니다.**
    - 시스템 프롬프트에 「팀 이름 줄을 기준으로 팀별로 나눈다, 팀 이름은 메모의 대괄호 안 글자를 그대로 쓴다, 담당자가 없으면 `owner`는 '미정', 지어내지 않는다」를 적고, `response_format`에 `WeeklyReport`를 지정해 모델을 한 번 호출합니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
4. **돌아온 문자열을 객체로 되돌리고 팀별 미결 목록을 만듭니다.**
    - `WeeklyReport.model_validate_json`으로 파싱하고, 팀마다 팀 이름·할 일 수·미결 수를 출력한 뒤, 모든 팀의 미결 항목을 「팀 이름: 미결」 형식의 목록 하나로 모읍니다.
5. **검증 실패를 관찰합니다.**
    - 안쪽 클래스의 필드가 빠진 입력(`teams.0.summary.decisions` 누락), 팀 이름이 겹치는 입력, 팀이 비어 있는 입력을 각각 `model_validate`에 넣어 오류 경로와 검증기 메시지를 관찰합니다.

## 5. 코드 골격 — 구조화 출력(pydantic) 4단

파이댄틱(pydantic)으로 구조화 출력을 받는 순서는 다음 네 단계입니다. 스키마가 중첩되어도 단계는 늘지 않습니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 스키마 선언 | 안쪽 클래스부터 바깥 클래스까지 선언하고 인스턴스 전체를 보는 검증기를 답니다 | `class WeeklyReport(BaseModel)`, `@model_validator(mode="after")` | 1, 2 |
| ② 스키마를 건 호출 | 모델 호출에 바깥 스키마를 지정해 출력 형식을 강제합니다 | `completion(..., response_format=WeeklyReport)` | 3 |
| ③ 객체 수신·파싱 | 돌아온 문자열을 클래스로 되돌려 팀별로 꺼내 씁니다 | `WeeklyReport.model_validate_json(...)` | 4 |
| ④ 검증 실패 관찰 | 안쪽 필드 누락·겹친 팀·빈 팀을 넣어 어디서 걸리는지 봅니다 | `ValidationError`, `model_validate(broken)` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import json
import os
import re
from datetime import date

from dotenv import load_dotenv, find_dotenv
from litellm import completion
from pydantic import BaseModel, ValidationError, field_validator, model_validator

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
print("준비를 마쳤습니다.")

# 주어진 자료 — 값과 이름을 그대로 씁니다.
MEMO = """9월 1주 주간 회의 메모

[개발팀]
- 다음 배포는 수요일로 확정했다.
- 접속 오류 재현 조건을 민수 씨가 다음 회의 전까지 정리하기로 했다.
- 서버 증설 예산은 결론을 내지 못했다.

[운영팀]
- 신규 문의 응대 템플릿을 이번 주부터 쓰기로 했다.
- 템플릿 초안은 지현 씨가 화요일까지 만들기로 했다.
- 야간 응대 인력 배치는 다음 주에 다시 논의한다.
- 매뉴얼 개편 범위는 결론을 내지 못했다."""


### 단계 ① — 스키마 선언 (요구사항 1, 2)

- 안쪽 클래스를 먼저 선언해야 바깥 클래스가 참조할 수 있습니다. `TeamSummary.summary`의 타입이 `MeetingSummary`이므로 안쪽 요약의 모양은 회의 하나의 요약과 같습니다.
- `@model_validator(mode="after")`는 필드 검증이 모두 끝난 뒤 인스턴스 하나를 통째로 받습니다. 필드 사이의 관계(겹침·빈 리스트)는 이 검증기에서만 볼 수 있습니다.
- 이 데코레이터를 붙인 메서드는 `self`를 받아 검사하고 마지막에 `self`를 돌려줍니다. `@field_validator`와 달리 `@classmethod`를 붙이지 않습니다.

In [ ]:
# 여기에 단계 ①(스키마 선언과 검증기)을 작성합니다.

### 단계 ② — 스키마를 건 호출 (요구사항 3)

- 스키마가 중첩되어도 호출은 한 번입니다. 안쪽 클래스의 모양이 바깥 스키마와 함께 모델에 전달됩니다.
- 이 셀은 모델을 한 번 호출합니다.


In [ ]:
# 여기에 단계 ②(추출 규칙, 입력 문서, 스키마를 지정한 호출 함수)를 작성합니다.

### 단계 ③ — 객체 수신·파싱 (요구사항 4)

- 파싱된 인스턴스에서 `report.teams[i].summary.action_items`처럼 점으로 안쪽까지 내려갑니다. 문자열이었다면 팀 경계를 다시 찾아야 했습니다.

In [ ]:
# 여기에 단계 ③(파싱과 다음 단계)을 작성합니다.

### 단계 ④ — 검증 실패 관찰 (요구사항 5)

- 안쪽 필드가 빠지면 오류 경로가 `teams.0.summary.decisions`처럼 바깥에서 안쪽으로 찍힙니다. 어느 팀의 어느 필드인지 경로만 보고 알 수 있습니다.
- 겹친 팀 이름과 빈 팀 리스트는 필드 검사를 통과하고 `@model_validator`에서 걸립니다.

In [ ]:
# 여기에 단계 ④(어긋난 입력으로 검증 실패 관찰)를 작성합니다.

## 7. 실행 결과 확인

1. 단계 ①의 출력에 바깥 스키마의 필드(`week`, `teams`)와 안쪽 클래스 `TeamSummary`·`MeetingSummary`의 필드가 찍힙니다.
2. 단계 ③에서 팀 수가 2이고, 개발팀과 운영팀의 이름 아래에 각각 할 일과 미결이 묶여 찍힙니다. 팀별 미결 목록에 두 팀의 미결이 팀 이름과 함께 모입니다.
3. 단계 ④의 첫 번째 오류 경로가 `teams.0.summary.decisions`이고, 두 번째와 세 번째 오류에 검증기 메시지(「팀 이름이 겹칩니다」「teams가 비어 있습니다」)가 찍힙니다.

확인 항목이 모두 맞으면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.